# Expérience 2 — similarité de contenu

**Deux réglages** à tester :

1. le **vivier de candidats** : tout le catalogue (364 047 articles) ou seulement
   les articles récents ?
2. la **longueur du profil** : moyenne des embeddings de *tous* les articles lus,
   ou seulement des `k` derniers ?

Le second point est spécifique à l'actualité : un centre d'intérêt d'il y a dix
jours ne vaut pas un centre d'intérêt d'hier, alors que la moyenne les traite à
égalité.

## Protocole commun

Identique dans tous les notebooks d'expérimentation, sinon les chiffres ne sont pas
comparables :

- **découpage temporel 60 / 20 / 20** sur `click_timestamp` ;
- artefacts construits sur la **seule** période d'entraînement (`models_split/`) ;
- réglage sur la **validation** ; la période de test reste intacte jusqu'à la mesure
  finale (notebook 07) ;
- lecteurs évalués : connus à l'entraînement **et** actifs pendant la période
  d'évaluation ;
- métriques : HitRate@5, Recall@5, couverture, personnalisation.

> Prérequis : `python -m src.evaluate --data-dir data/news-portal-user --out-dir models_split`

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.append('..')

from src import experiments as xp
from src.recommender import Recommender

DATA = Path('..') / 'data' / 'news-portal-user'
MODELS = Path('..') / 'models_split'

train, val, test = xp.load_split(DATA)
reco = Recommender(MODELS)
users, cible = xp.eval_users(reco, val, max_users=2000)
print(f'{len(users):,} lecteurs évalués sur la période de validation')

[clicks] 1 fichier(s) vide(s) ignoré(s) : clicks_hour_100.csv


[split] entraînement 1,792,908 | validation 597,636 | test 597,637
[split] bornes temporelles : t60=1507602953792 t80=1507843212579


2,000 lecteurs évalués sur la période de validation


## 1. Effet du vivier de candidats

Même méthode, seul le vivier change.

In [2]:
catalogue_entier = np.arange(reco.n_articles, dtype=np.int64)

configs = {'tout le catalogue': xp.make_content(reco, catalogue_entier)}
for heures in (240, 72, 24, 6, 3, 1):
    configs[f'récents {heures} h'] = xp.make_content(reco, xp.recent_pool(train, heures))

xp.compare(configs, users, cible, n_articles=reco.n_articles)

,HitRate@5,Recall@5,couverture %,personnalisation %
configuration,,,,
tout le catalogue,0.0020,0.0003,1.452,99.8
récents 240 h,0.0065,0.0013,0.794,98.6
récents 72 h,0.0210,0.0032,0.563,98.3
récents 24 h,0.0295,0.0044,0.468,98.2
récents 6 h,0.0405,0.0059,0.313,95.1
récents 3 h,0.0360,0.0047,0.246,94.3
récents 1 h,0.0255,0.0031,0.158,92.8


## 2. Effet de la longueur du profil

À vivier fixé (la meilleure fenêtre ci-dessus), on fait varier le nombre d'articles
retenus pour construire le profil.

In [3]:
# 6 h est l'optimum mesuré : un pool plus court (1 h, 791 articles) prive le
# contenu de candidats et fait chuter le HitRate à 0,0235.
MEILLEURE_FENETRE = 6
pool = xp.recent_pool(train, MEILLEURE_FENETRE)

configs = {'profil complet': xp.make_content(reco, pool, last_k=None)}
for k in (1, 3, 5, 10, 20):
    configs[f'{k} derniers articles'] = xp.make_content(reco, pool, last_k=k)

xp.compare(configs, users, cible, n_articles=reco.n_articles)

,HitRate@5,Recall@5,couverture %,personnalisation %
configuration,,,,
profil complet,0.0405,0.0059,0.313,95.1
1 derniers articles,0.0300,0.0039,0.412,99.4
3 derniers articles,0.0245,0.0037,0.467,98.2
5 derniers articles,0.0285,0.0043,0.436,97.6
10 derniers articles,0.0360,0.0060,0.367,97.1
20 derniers articles,0.0385,0.0059,0.336,96.1


## Lecture

Le vivier est le levier principal : sans restriction, la méthode propose des
articles vieux de plusieurs années, similaires mais illisibles par personne — d'où
un HitRate proche de zéro et, paradoxalement, la **meilleure couverture** de toutes
les méthodes.

C'est le point à retenir sur cette méthode : sa valeur n'est pas la précision mais
la **diversité**. Elle expose une part du catalogue deux ordres de grandeur plus
grande que la popularité, avec une personnalisation proche de 100 %. Pour un
éditeur qui ne veut pas que seuls cinq articles soient lus, c'est un critère réel,
que le HitRate ne mesure pas.